<a href="https://colab.research.google.com/github/Kevoharvey/DermaXRAI/blob/feat%2Fcleanandprocessingdata/testing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Derm1M: clean data, split it, and prepare a ResNet

This is a simple, one-notebook path from raw Derm1M metadata to files that a
**ResNet-18** can use. There is no Transformer model in this notebook.

Think of the steps as: **read → clean → label → keep friends together → split
→ prepare pictures → save**.


## Before you start

- Accept the Derm1M terms on Hugging Face, or set `DERM1M_MANIFEST` to a local
  copy of `Derm1M_v2_pretrain.csv`.
- Set `DERM1M_IMAGE_ROOT` to the extracted image folder before the final run.
  That lets us find missing/broken pictures and exact copies.
- Run the cells from top to bottom. The finished files go in `dataset/processed/`.

We use the word `steps` for image preparation below. Those are simple image
edits such as resize and crop; they are **not** Transformer models.


In [1]:
# The tools we need. Install missing packages with:
# pip install pandas numpy scikit-learn pillow torch torchvision huggingface_hub
import hashlib
import json
import os
import re
from itertools import combinations
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image, UnidentifiedImageError
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.utils.class_weight import compute_class_weight

# Find the main project folder.
project = Path.cwd().resolve()
if not (project / "model").is_dir():
    project = project.parent

# These are the only settings most students need to change.
seed = 42
image_size = 224
save_folder = project / "dataset" / "processed"
image_folder = Path(os.environ["DERM1M_IMAGE_ROOT"]) if os.getenv("DERM1M_IMAGE_ROOT") else None

# The order here is the number the ResNet learns: acne=0, eczema=1, and so on.
class_names = ["acne", "eczema", "psoriasis", "melanoma", "benign_nevus", "fungal_infection"]
number_for = {name: number for number, name in enumerate(class_names)}

# These are approved Derm1M diagnosis words. Add a word only after a clinical review.
words_for = {
    "acne": {"acne", "acne vulgaris", "acne conglobata", "acne inversa", "acneiform eruption", "comedonal acne", "cystic acne"},
    "eczema": {"eczema", "atopic dermatitis", "atopic eczema", "contact dermatitis", "allergic contact dermatitis", "irritant contact dermatitis", "dyshidrotic eczema", "nummular eczema", "hand eczema"},
    "psoriasis": {"psoriasis", "plaque psoriasis", "guttate psoriasis", "inverse psoriasis", "pustular psoriasis", "scalp psoriasis", "psoriatic erythroderma"},
    "melanoma": {"melanoma", "malignant melanoma", "cutaneous melanoma", "acral lentiginous melanoma", "lentigo maligna melanoma", "superficial spreading melanoma", "nodular melanoma"},
    "benign_nevus": {"nevus", "naevus", "melanocytic nevus", "melanocytic naevus", "benign nevus", "benign naevus", "compound nevus", "junctional nevus", "intradermal nevus", "congenital melanocytic nevus", "blue nevus", "dysplastic nevus", "spitz nevus"},
    "fungal_infection": {"tinea", "tinea corporis", "tinea cruris", "tinea pedis", "tinea versicolor", "pityriasis versicolor", "onychomycosis", "dermatophytosis", "candidiasis", "cutaneous candidiasis"},
}
ignore_words = {"dermatitis", "nail disease"}

# Make one easy lookup table: "acne vulgaris" -> "acne".
word_to_class = {}
for class_name, words in words_for.items():
    for word in words:
        word = " ".join(word.lower().split())
        if word in word_to_class:
            raise ValueError(f"This word is used twice: {word}")
        word_to_class[word] = class_name

print("Classes:", class_names)
print("Picture check:", "on" if image_folder else "off — set DERM1M_IMAGE_ROOT before training")


Classes: ['acne', 'eczema', 'psoriasis', 'melanoma', 'benign_nevus', 'fungal_infection']
Picture check: off — set DERM1M_IMAGE_ROOT before training


## 1. Read the spreadsheet

If `DERM1M_MANIFEST` is set, we read that local CSV. Otherwise, Hugging Face
gives us the Derm1M metadata CSV. We only need three columns.


In [2]:
# Sign in only when we need to download the gated Derm1M CSV.
# Paste a Hugging Face READ token when asked, or set HF_TOKEN before opening Jupyter.
if not os.getenv("DERM1M_MANIFEST"):
    from huggingface_hub import login
    token = os.getenv("HF_TOKEN")
    login(token=token) if token else login()
    print("Signed in. If this is your first time, also accept Derm1M access terms on Hugging Face.")


Signed in. If this is your first time, also accept Derm1M access terms on Hugging Face.


In [3]:
local_csv = os.getenv("DERM1M_MANIFEST")
if local_csv:
    csv_file = Path(local_csv).expanduser()
    if not csv_file.is_file():
        raise FileNotFoundError(f"Cannot find: {csv_file}")
else:
    from huggingface_hub import hf_hub_download
    csv_file = Path(hf_hub_download("redlessone/Derm1M", "Derm1M_v2_pretrain.csv", repo_type="dataset"))

data = pd.read_csv(csv_file, encoding="utf-8-sig", low_memory=False)
needed_columns = {"filename", "disease_label", "source"}
if not needed_columns.issubset(data.columns):
    raise ValueError(f"The CSV needs these columns: {sorted(needed_columns)}")
data = data[["filename", "disease_label", "source"]].copy()
print(f"We read {len(data):,} rows.")
data.head()


Derm1M_v2_pretrain.csv: reconstructing file:   0%|          |  0.00B /  323MB            

Derm1M_v2_pretrain.csv: downloading bytes:           |  0.00B            

We read 413,210 rows.


,filename,disease_label,source
0,IIYI/26549_2.png,epidermolysis bullosa,IIYI_chinese
1,youtube/ykOs8g7zjDc_frame_1135_15.jpg,no definitive diagnosis,youtube
2,youtube/8wBKdJ1JWHQ_frame_6758_0_0.jpg,"erythrasma, pitriasis versicolor, tenia capiti...",youtube
3,pubmed/b9_2b_PMC4886598_IDOJ_7_203_g001.png,ulcer,pubmed_english
4,youtube/3bLlU_wTljA_frame_32617_3.jpg,nevus,youtube


## 2. Clean the spreadsheet

We remove empty rows, exact copies of a row, and files that have two different
diagnosis strings. A picture cannot teach the model two different answers.


In [4]:
for column in ["filename", "disease_label", "source"]:
    data[column] = data[column].fillna("").astype(str).str.strip()
data["filename"] = data["filename"].str.replace("\\", "/", regex=False)

empty_row = data.filename.eq("") | data.disease_label.eq("")
copied_row = data.duplicated()
mixed_files = (
    data.loc[~empty_row].groupby("filename").disease_label.nunique()
    .loc[lambda count: count > 1].index
)

data = data.loc[~empty_row & ~copied_row & ~data.filename.isin(mixed_files)].copy()
data = data.drop_duplicates("filename").reset_index(drop=True)

print("Empty rows removed:", int(empty_row.sum()))
print("Copied rows removed:", int(copied_row.sum()))
print("Files with mixed labels removed:", len(mixed_files))
print("Rows left:", len(data))


Empty rows removed: 0
Copied rows removed: 0
Files with mixed labels removed: 0
Rows left: 413210


## 3. Give each picture one safe class name

A Derm1M label can have several comma-separated words. We keep a row only if
all useful words point to the **same** class. Unknown words and mixed classes
are left out instead of guessed.


In [5]:
def clean_word(word):
    return " ".join(str(word).lower().strip().split())

def pick_class(label_text):
    found_classes = []
    for word in str(label_text).split(","):
        word = clean_word(word)
        if not word or word in ignore_words:
            continue
        if word not in word_to_class:
            return None
        found_classes.append(word_to_class[word])

    if not found_classes or len(set(found_classes)) != 1:
        return None
    return found_classes[0]

# Save unknown words before we remove their rows, so a human can review them.
all_words = data.disease_label.str.split(",").explode().map(clean_word)
review_words = all_words[~all_words.isin(word_to_class) & ~all_words.isin(ignore_words)]

data["label"] = data.disease_label.map(pick_class)
skipped = int(data.label.isna().sum())
data = data.dropna(subset=["label"]).reset_index(drop=True)

# We want enough examples for every class.
class_count = data.label.value_counts()
class_names = [name for name in class_names if class_count.get(name, 0) >= 500]
number_for = {name: number for number, name in enumerate(class_names)}
data = data[data.label.isin(class_names)].reset_index(drop=True)

print("Rows skipped because the label was not safe:", skipped)
display(data.label.value_counts().reindex(class_names).rename("pictures").to_frame())


Rows skipped because the label was not safe: 307152


,pictures
label,
acne,2745
eczema,63089
psoriasis,14460
melanoma,8091
benign_nevus,10649
fungal_infection,7024


In [6]:
# Show a small list of unknown label words for a human to review later.
review_words.value_counts().rename_axis("word").reset_index(name="times_seen").head(20)


,word,times_seen
0,no definitive diagnosis,102696
1,"irritated seborrheic keratosis (from ""sk/isk"")",17864
2,keratosis,8091
3,basal cell carcinoma,6251
4,lichen planus,4307
5,squamous cell carcinoma,3578
6,urticaria,3499
7,proliferations,3448
8,ulcer,2702
9,skin diseases caused by warts,2592


## 4. Keep related pictures together

A video can contain many frames of the same person. We give related pictures
the same group name, so they cannot land in different splits. A picture with
no clear family gets its own group.


In [7]:
def family_name(file_name):
    path = str(file_name).replace("\\", "/").lstrip("/").lower()
    pieces = path.split("/")

    # Video, case, and social-post folders usually use the next folder as an ID.
    if len(pieces) >= 2 and pieces[0] in {"youtube", "iiyi", "twitter", "reddit"}:
        return f"{pieces[0]}:{pieces[1]}"

    # Pictures from one PubMed article stay together.
    article = re.search(r"\b(pmc\d+)\b", path)
    if article:
        return f"pubmed:{article.group(1)}"

    # Textbook crops from the same page stay together.
    if pieces[0] in {"edu", "note"}:
        page = re.sub(r"(?:[_-](?:fig(?:ure)?|image)?\d+[a-z]?)$", "", Path(path).stem)
        return f"book:{Path(path).parent}/{page}"

    return f"one_picture:{path}"

data["group"] = data.filename.map(family_name)
print("Groups:", f"{data.group.nunique():,}")
display(data.groupby("label").group.nunique().reindex(class_names).rename("groups").to_frame())


Groups: 104,422


,groups
label,
acne,2631
eczema,62790
psoriasis,14209
melanoma,7868
benign_nevus,10246
fungal_infection,6758


## 5. Check picture files (optional, but required before training)

This works only after `DERM1M_IMAGE_ROOT` is set. It removes missing/broken
pictures. Exact duplicate files get the same group, so a copy cannot leak into
the test set. If the folder is not set, this cell politely skips the check.


In [8]:
if image_folder is None:
    print("Skipped. Set DERM1M_IMAGE_ROOT, then rerun before training.")
elif not image_folder.is_dir():
    raise NotADirectoryError(f"Not a folder: {image_folder}")
else:
    good = []
    hashes = []
    for file_name in data.filename:
        picture = image_folder / file_name.lstrip("/")
        try:
            with Image.open(picture) as opened_picture:
                opened_picture.verify()
            good.append(True)
            hashes.append(hashlib.sha256(picture.read_bytes()).hexdigest())
        except (FileNotFoundError, OSError, UnidentifiedImageError, ValueError):
            good.append(False)
            hashes.append(None)

    data["good_picture"] = good
    data["file_hash"] = hashes
    print("Missing or broken pictures removed:", int((~data.good_picture).sum()))
    data = data[data.good_picture].copy()

    # A same-byte picture with two labels is unsafe, so remove both labels.
    bad_hashes = data.groupby("file_hash").label.nunique().loc[lambda count: count > 1].index
    data = data[~data.file_hash.isin(bad_hashes)].copy()
    copied_picture = data.file_hash.duplicated(keep=False)
    data.loc[copied_picture, "group"] = "same_file:" + data.loc[copied_picture, "file_hash"]
    print("Exact-copy groups with mixed labels removed:", len(bad_hashes))


Skipped. Set DERM1M_IMAGE_ROOT, then rerun before training.


## 6. Make train, validation, and test groups

We make 20 small, mixed groups. Three go to test, three go to validation, and
fourteen go to training: about **15% / 15% / 70%**. The random seed makes the
result repeatable.


In [9]:
folds = 20
groups_per_class = data.groupby("label").group.nunique()
if groups_per_class.min() < folds:
    raise ValueError("A class needs at least 20 groups before this split can work.")

splitter = StratifiedGroupKFold(n_splits=folds, shuffle=True, random_state=seed)
data["fold"] = -1
for fold_number, (_, test_rows) in enumerate(splitter.split(data, y=data.label, groups=data.group)):
    data.loc[data.index[test_rows], "fold"] = fold_number

data["split"] = "train"
data.loc[data.fold.isin([0, 1, 2]), "split"] = "test"
data.loc[data.fold.isin([3, 4, 5]), "split"] = "val"
display(data.split.value_counts().reindex(["train", "val", "test"]).rename("pictures").to_frame())


,pictures
split,
train,74276
val,15880
test,15902


In [10]:
# This is our safety check: no group or file may appear in two places.
checks = []
for first, second in combinations(["train", "val", "test"], 2):
    first_data = data[data.split.eq(first)]
    second_data = data[data.split.eq(second)]
    checks.append({
        "check": f"{first} and {second}",
        "same groups": len(set(first_data.group) & set(second_data.group)),
        "same files": len(set(first_data.filename) & set(second_data.filename)),
    })
safety_check = pd.DataFrame(checks).set_index("check")
assert not safety_check.to_numpy().any(), f"Split leak found!\n{safety_check}"
print("Great: no group or file leaks between the three splits.")
display(safety_check)


Great: no group or file leaks between the three splits.


,same groups,same files
check,,
train and val,0,0
train and test,0,0
val and test,0,0


## 7. Balance the training loss

We do **not** copy rare pictures or change validation/test. Instead, rare
training classes get a slightly louder voice in the loss function.


In [11]:
split_table = pd.crosstab(data.label, data.split).reindex(index=class_names, columns=["train", "val", "test"], fill_value=0)
display(split_table)

train_labels = data.loc[data.split.eq("train"), "label"]
weights = compute_class_weight("balanced", classes=np.array(class_names), y=train_labels)
weights = {name: float(weight) for name, weight in zip(class_names, weights)}
display(pd.DataFrame({"train pictures": train_labels.value_counts().reindex(class_names), "loss weight": pd.Series(weights)}))


split,train,val,test
label,,,
acne,1960,392,393
eczema,44066,9517,9506
psoriasis,10212,2119,2129
melanoma,5636,1209,1246
benign_nevus,7535,1572,1542
fungal_infection,4867,1071,1086


,train pictures,loss weight
acne,1960,6.315986
eczema,44066,0.280927
psoriasis,10212,1.212234
melanoma,5636,2.196475
benign_nevus,7535,1.642911
fungal_infection,4867,2.543524


## 8. Prepare pictures and create a ResNet-18

This is an ordinary **convolutional ResNet-18**, not a Transformer. Training
pictures get tiny, safe changes to help the model learn. Validation and test
pictures get the same fixed steps every time.


In [12]:
import torch
from torch import nn
from torch.utils.data import Dataset
from torchvision import transforms
from torchvision.models import ResNet18_Weights, resnet18

# Image preparation steps for a pretrained ResNet.
train_steps = transforms.Compose([
    transforms.Resize(256),
    transforms.RandomResizedCrop(image_size, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225)),
])
test_steps = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(image_size),
    transforms.ToTensor(),
    transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225)),
])

class SkinPictures(Dataset):
    def __init__(self, rows, steps):
        if image_folder is None:
            raise ValueError("Set DERM1M_IMAGE_ROOT before making picture datasets.")
        self.rows = rows.reset_index(drop=True)
        self.steps = steps

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, row_number):
        row = self.rows.iloc[row_number]
        with Image.open(image_folder / row.filename.lstrip("/")) as picture:
            picture = self.steps(picture.convert("RGB"))
        return picture, number_for[row.label]

# Start with a small, common ResNet and replace its final answer layer.
# The first option uses saved ImageNet knowledge. The second still works offline.
try:
    model = resnet18(weights=ResNet18_Weights.DEFAULT)
    print("Using pretrained ResNet-18 weights.")
except Exception:
    print("Pretrained weights are not available, so we will train ResNet-18 from scratch.")
    model = resnet18(weights=None)
model.fc = nn.Linear(model.fc.in_features, len(class_names))
loss_function = nn.CrossEntropyLoss(weight=torch.tensor([weights[name] for name in class_names], dtype=torch.float32))
optimizer = torch.optim.Adam(model.parameters(), lr=0.0003)

print("Model:", model.__class__.__name__)
print("Final answers:", len(class_names))


Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 89.6MB/s]


Using pretrained ResNet-18 weights.
Model: ResNet
Final answers: 6


## 9. Save the clean result

Later training code should read these files, not the raw Derm1M CSV. Use
`train.csv` to learn, `val.csv` to choose settings, and `test.csv` only for
the final score.


In [13]:
save_folder.mkdir(parents=True, exist_ok=True)
columns_to_save = ["filename", "label", "source", "group", "fold", "split"]
if "file_hash" in data.columns:
    columns_to_save.append("file_hash")

data[columns_to_save].sort_values(["split", "label", "filename"]).to_csv(save_folder / "cleaned_manifest.csv", index=False)
for split_name in ["train", "val", "test"]:
    data.loc[data.split.eq(split_name), columns_to_save].sort_values(["label", "filename"]).to_csv(
        save_folder / f"{split_name}.csv", index=False
    )

review_words.value_counts().rename_axis("word").reset_index(name="times_seen").to_csv(
    save_folder / "unmapped_words.csv", index=False
)
with (save_folder / "class_weights_train.json").open("w") as file:
    json.dump(weights, file, indent=2)
with (save_folder / "settings.json").open("w") as file:
    json.dump({
        "seed": seed,
        "image_size": image_size,
        "classes": class_names,
        "class_words": {name: sorted(words_for[name]) for name in class_names},
        "csv_sha256": hashlib.sha256(csv_file.read_bytes()).hexdigest(),
        "split_sizes": data.split.value_counts().to_dict(),
        "used_picture_check": image_folder is not None,
        "model": "ResNet-18",
    }, file, indent=2)

print(f"Saved clean files in: {save_folder}")


Saved clean files in: /dataset/processed


## Tiny checklist

1. Look at `unmapped_words.csv` before adding any new diagnosis word.
2. Rerun with `DERM1M_IMAGE_ROOT` set before training.
3. Train the ResNet using `train.csv`, choose with `val.csv`, and keep
   `test.csv` for the final report only.
